# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [118]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [119]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

In [120]:
import os
import json
import importlib.util
import sys
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
from typing import TypedDict
from lib.state_machine import StateMachine, Step, EntryPoint, Termination

In [121]:
# TODO: Load environment variables
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [122]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# Connect to the same persistent Chroma database
chroma_client = chromadb.PersistentClient(path="chroma_db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [123]:
print(chroma_client.list_collections())
 

[Collection(name=games)]


In [124]:
print("Embedding function ready")

Embedding function ready


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [125]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

def retrieve_game(query: str):
    """
    Retrieve relevant games from the game knowledge base.
 
    Args:
        query: The user's search query.
 
    Returns:
        A list of matching game documents and metadata.
    """
 
    # Connect to the Chroma database created in Part 1
    #chroma_client = chromadb.PersistentClient(path="chroma_db")
 
    # Get the existing games collection
    collection = chroma_client.get_collection(
        name="games"
    )
 
    # Chroma genertes the query embedding automatically
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    #query_embedding = embedding_fn([query])
 
    # Search Chroma using the generated embedding
    #results = collection.query(
    #    query_embeddings=query_embedding,
    #   n_results=3
    #)
 
    retrieved_games = []
 
    for document, metadata in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):
        retrieved_games.append({
            "document": document,
            "metadata": metadata
        })
 
    return retrieved_games

In [126]:
import os
from openai import OpenAI
 
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

print("Client created successfully")

Client created successfully


#### Evaluate Retrieval Tool

In [127]:
def evaluate_retrieval(question, retrieved_docs):
    """
    Evaluate whether the retrieved documents are sufficient
    to answer the user's question.
    """
 
    # No documents retrieved
    if not retrieved_docs:
        return {
            "useful": False,
            "description": "No relevant documents were retrieved."
        }
 
    # Extract words from the question
    question_words = {
        word.lower().strip("?!.,")
        for word in question.split()
        if len(word) > 2
    }
 
    # Combine retrieved document text
    documents_text = " ".join(
        doc.get("document", "")
        for doc in retrieved_docs
    ).lower()
 
    # Check whether important question terms occur
    matching_words = [
        word for word in question_words
        if word in documents_text
    ]
 
    useful = len(matching_words) > 0
 
    if useful:
        description = (
            "Retrieved relevant documents are sufficient "
            "to help answer the user's question."
        )
    else:
        description = (
            "Retrieved documents do not contain enough "
            "relevant information to answer the user's question."
        )
 
    return {
        "useful": useful,
        "description": description
    }

In [128]:
question = "Who developed FIFA 21?"
 
retrieved_docs = retrieve_game(question)
 
evaluation = evaluate_retrieval(
    question,
    retrieved_docs
)
 
print(evaluation)

{'useful': False, 'description': "Retrieved documents do not contain enough relevant information to answer the user's question."}


#### Game Web Search Tool

In [129]:
from tavily import TavilyClient
import os
 
def game_web_search(question):
    tavily_client = TavilyClient(
        api_key=os.getenv("TAVILY_API_KEY")
    )
 
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5
    )
 
    return response["results"]

In [130]:
web_results = game_web_search("Who developed FIFA 21?")
 
for result in web_results:
    print("Title:", result.get("title"))
    print("Content:", result.get("content"))
    print("URL:", result.get("url"))
    print("--------------------")

Title: FIFA 21 - Simple English Wikipedia, the free encyclopedia
Content: FIFA 21 is a 2020 association football simulation video game in Electronic Arts' FIFA "FIFA (video game series)") series created by EA Vancouver and published by Electronic Arts. It was released for the Playstation 4, PlayStation 5, Xbox One, Xbox Series X and Series S, Nintendo Switch, Google Stadia and Microsoft Windows.

  e  Retired Players in FIFA "FIFA (video game series)") Ultimate Team (Since FIFA 18) | [...] Wikimedia Commons
 Wikidata item

Appearance

From Simple English Wikipedia, the free encyclopedia

| FIFA 21 |

| FIFA 21 logo |
| Developer(s) | EA Vancouver EA Romania |
| Publisher(s) | EA Sports |
| Series | FIFA "FIFA (video game series)") |
| Engine | Frostbite 3?action=edit&redlink=1 "Frostbite (game engine) (not yet started)") |
| Platform(s) |  Microsoft Windows  PlayStation 4  Xbox One  Nintendo Switch  Stadia  PlayStation 5  Xbox Series X/S |
| Release |  Microsoft Windows, Nintendo Switc

In [131]:
question = "Who developed FIFA 21?"
retrieved_docs = retrieve_game(question)
 
evaluation = evaluate_retrieval(
    question,
    retrieved_docs
)
 
print(evaluation)
print(type(evaluation))

{'useful': False, 'description': "Retrieved documents do not contain enough relevant information to answer the user's question."}
<class 'dict'>


### Agent

In [132]:
class AgentState(TypedDict):
    question: str
    retrieved_docs: list
    evaluation: object
    web_results: list
    answer: str
 
 
workflow = StateMachine(AgentState)
 
 
# Step 1 - Retrieve from Vector DB
def retrieve_step(state: AgentState):
    docs = retrieve_game(state["question"])
 
    return {
        "retrieved_docs": docs
    }
 
 
# Step 2 - Evaluate retrieved documents
def evaluate_step(state: AgentState):
    evaluation = evaluate_retrieval(
        state["question"],
        state["retrieved_docs"]
    )
 
    return {
        "evaluation": evaluation
    }
 
 
# Step 3 - Answer using Vector DB results
def vector_answer_step(state: AgentState):
 
    docs = state["retrieved_docs"]
 
    answer = "\n".join(
        [str(doc) for doc in docs]
    )
 
    return {
        "answer": answer
    }
 
 
# Step 4 - Search web if Vector DB was not useful
def web_search_step(state: AgentState):
 
    results = game_web_search(
        state["question"]
    )
 
    return {
        "web_results": results,
        "answer": str(results)
    }
 
 
# Create workflow components
 
entry = EntryPoint()
 
retrieve_node = Step(
    "retrieve_game",
    retrieve_step
)
 
evaluate_node = Step(
    "evaluate_retrieval",
    evaluate_step
)
 
vector_answer_node = Step(
    "vector_answer",
    vector_answer_step
)
 
web_search_node = Step(
    "web_search",
    web_search_step
)
 
termination = Termination()
 
 
workflow.add_steps([
    entry,
    retrieve_node,
    evaluate_node,
    vector_answer_node,
    web_search_node,
    termination
])
 
 
workflow.connect(
    entry,
    retrieve_node
)
 
workflow.connect(
    retrieve_node,
    evaluate_node
)
 

In [133]:
# Router
from types import SimpleNamespace
 
def route_after_evaluation(state: AgentState):
    evaluation = state["evaluation"]
 
    if evaluation["useful"]:
        return [vector_answer_node]
 
    return [web_search_node]

In [134]:
# Connections
workflow.connect(
    evaluate_node,
    [vector_answer_node, web_search_node],
    route_after_evaluation
)
 
workflow.connect(
    vector_answer_node,
    termination
)
 
workflow.connect(
    web_search_node,
    termination
)

In [138]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
# Test Agent
initial_state = {
    "question": "When Pokemon Gold and Silver was released?",
    "retrieved_docs": [],
    "evaluation": None,
    "web_results": [],
    "answer": ""
}
 
run_object = workflow.run(initial_state)
 
print(run_object)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: vector_answer
[StateMachine] Terminating: __termination__
Run('68bad5ba-592b-4982-be20-a3ec47cf8d80')


### (Optional) Advanced

In [137]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes